# 대화 기억 RAG 실습

사용자 발언·NPC 응답·엔진 공개 사건을 구분해 저장하고 현재 질문과 가까운 기록을 찾습니다.
**CPU 런타임에서 위부터 실행**합니다. 생성 모델과 GPU는 필요하지 않습니다.
처음 패키지·임베딩 모델을 받을 때는 인터넷 연결이 필요합니다.

예제는 모두 가상의 대화·엔진 결과입니다. 실제 모델 응답이나 성능 점수를 저장한 자료가 아닙니다.
**누군가 말했다는 기록과 실제로 일어난 사건은 다릅니다.**
검색 점수는 문장 유사도이며 진실 여부나 정답률이 아닙니다.
기억은 런타임 안의 임시 DB에 저장하며 파일로 영구 저장하지 않습니다.


## 0. 라이브러리 설치

Chroma는 벡터 저장·검색, Sentence Transformers는 임베딩에 사용합니다.
런타임의 PyTorch를 사용하며 이 셀에서 교체하지 않습니다.
재시작 안내가 나오면 런타임을 재시작하고 위부터 실행합니다.


In [ ]:
%pip install -q "chromadb>=1,<2" "sentence-transformers>=5,<6"


## 1. CPU 임베딩 모델 준비

다국어 `intfloat/multilingual-e5-small`을 고정 revision으로 준비합니다.
저장 문장 앞의 `passage:`와 질문 앞의 `query:`는 기억 함수가 붙입니다.
모델 파일 캐시와 이후 저장할 대화 기억은 서로 다른 자료입니다.


In [ ]:
from sentence_transformers import SentenceTransformer

EMBEDDING_MODEL = "intfloat/multilingual-e5-small"
EMBEDDING_REVISION = "614241f622f53c4eeff9890bdc4f31cfecc418b3"
embedder = SentenceTransformer(
    EMBEDDING_MODEL, revision=EMBEDDING_REVISION,
    device="cpu", trust_remote_code=False,
)
print("CPU 임베딩 준비 완료")


## 2. 기억 모듈 정의

아래 셀은 출력 프로토타입과 같은 전체 모듈입니다. 별도 Python 파일 없이 실행합니다.

| 함수 | 하는 일 |
|---|---|
| `create_memory_session` | 접속별 고유 세션 생성 |
| `remember_player_message` | 사용자 발언 저장·턴 번호 반환 |
| `remember_npc_message` | 같은 턴의 NPC 응답·서술을 미확정 기록으로 저장 |
| `remember_confirmed_event` | 신뢰할 수 있는 엔진의 공개 사건 저장 |
| `save/load_dialogue_history` | NPC별 구조화된 이력 저장·복원 |
| `search_memory` | 같은 세션·NPC 범위의 기록 검색 |
| `discard_memory_session` | 해당 세션의 모든 NPC 기록 폐기 |

발언은 `unverified_statement`, 엔진 사건만 `engine_confirmed`입니다.
생성된 응답을 엔진 사건 저장 함수에 넣으면 안 됩니다.


In [ ]:
# %% 4. session memory RAG
"""세션·NPC별 대화와 확인된 공개 사건을 구분하는 임시 기억 저장소."""
from copy import deepcopy
from threading import RLock
from uuid import uuid4

import chromadb


# 프로세스 수명 동안 사용하는 Chroma 임시 기억 DB입니다.
_memory_client = None
_memory_collection = None
_memory_collection_name = "trpg_memory_" + uuid4().hex
_memory_sessions = {}
_memory_lock = RLock()  # 동시 입력 시 같은 NPC의 턴 번호가 겹치지 않게 합니다.


def _memory_store():
    """필요할 때 Chroma 임시 컬렉션을 한 번 생성합니다."""
    global _memory_client, _memory_collection
    if _memory_collection is None:
        # 같은 프로세스의 검색 모듈과 기본 설정을 공유합니다.
        # 데이터 범위는 고유 컬렉션 이름과 세션·NPC 필터로 분리합니다.
        _memory_client = chromadb.EphemeralClient()
        _memory_collection = _memory_client.create_collection(
            name=_memory_collection_name,
            embedding_function=None,
            metadata={"hnsw:space": "cosine"},
        )
    return _memory_collection


def create_memory_session():
    """첫 요청·초기화 시 새 ID를 만들고 Gradio State에 보관합니다."""
    session_id = uuid4().hex
    with _memory_lock:
        _memory_sessions[session_id] = {"scopes": {}}
    return session_id


def _memory_scope(session_id, npc):
    """알려진 세션만 허용하고, 비어 있는 인물명은 GM 범위로 통일합니다."""
    if not isinstance(session_id, str) or session_id not in _memory_sessions:
        raise ValueError("기억 세션이 없습니다. 새 대화를 시작하세요.")
    if not isinstance(npc, str):
        raise ValueError("대화 상대는 문자열이어야 합니다.")
    return npc.strip() or "게임 마스터"


def _memory_where(session_id, npc):
    """기억 조회·검색·삭제에 항상 세션과 대화 상대를 함께 적용합니다."""
    return {"$and": [{"session_id": {"$eq": session_id}},
                     {"npc": {"$eq": npc}}]}


def _memory_text(text):
    if not isinstance(text, str) or not text.strip():
        raise ValueError("기억에 저장할 문장이 필요합니다.")
    return text.strip()


def _npc_memory_scope(session_id, npc):
    """검색 기록과 화면 이력이 같은 NPC 범위를 사용하도록 합니다. 잠금 안에서 호출합니다."""
    npc = _memory_scope(session_id, npc)
    scopes = _memory_sessions[session_id]["scopes"]
    return npc, scopes.setdefault(npc, {"player_turn": 0, "event_index": 0,
                                       "dialogue_history": []})


def load_dialogue_history(session_id, npc):
    """해당 NPC와의 이력을 복사해 반환합니다. 다른 NPC로 전환해도 지우지 않습니다."""
    with _memory_lock:
        npc = _memory_scope(session_id, npc)
        scope = _memory_sessions[session_id]["scopes"].get(npc, {})
        return deepcopy(scope.get("dialogue_history", []))


def save_dialogue_history(session_id, npc, history):
    """화면 이력을 복사해 보관합니다. 대화 저장은 세계 상태를 확정하지 않습니다.

    role/content 외에 memory_turn, answered_by, speech, narration 같은 필드도 유지합니다.
    반환한 이력이나 원래 리스트를 수정해도 저장된 이력은 바뀌지 않습니다.
    """
    if not isinstance(history, list):
        raise ValueError("대화 이력은 메시지 목록이어야 합니다.")
    copied = deepcopy(history)
    for item in copied:
        if (not isinstance(item, dict) or item.get("role") not in {"user", "assistant"}
                or not isinstance(item.get("content"), str)):
            raise ValueError("대화 이력에는 user/assistant 역할과 문자열 content가 필요합니다.")
        if "memory_turn" in item and (type(item["memory_turn"]) is not int
                                      or item["memory_turn"] < 0):
            raise ValueError("memory_turn은 0 이상의 정수여야 합니다.")
        for field in ("speech", "narration"):
            if field in item and not isinstance(item[field], str):
                raise ValueError(f"{field}는 문자열이어야 합니다.")
    with _memory_lock:
        _, scope = _npc_memory_scope(session_id, npc)
        scope["dialogue_history"] = copied


def _remember_memory(session_id, npc, text, embedder, kind, turn_id=None,
                     speech=None, narration=None):
    """본문을 E5로 임베딩하고 세션·NPC·기록 종류와 함께 저장합니다."""
    with _memory_lock:
        npc = _memory_scope(session_id, npc)
    text = _memory_text(text)
    if kind not in {"player_statement", "npc_statement", "confirmed_event"}:
        raise ValueError("지원하지 않는 기억 기록 종류입니다.")
    if turn_id is not None and (type(turn_id) is not int or turn_id < 0):
        raise ValueError("발언 번호는 0 이상의 정수여야 합니다.")
    for value in (speech, narration):
        if value is not None and not isinstance(value, str):
            raise ValueError("대사와 서술은 문자열이어야 합니다.")
    if kind == "player_statement":
        document = (f"발언자=플레이어, 청자=NPC {npc}, 확인되지 않은 발언.\n"
                    f"플레이어가 말한 원문: 「{text}」\n"
                    "인용문 속 나/저/제는 플레이어를 가리키며 NPC의 신원이나 설정이 아닙니다.")
        trust = "unverified_statement"
        speaker = "player"
        listener = npc
        role = "user"
    elif kind == "npc_statement":
        document = f"대화 상대=NPC {npc}, 청자=플레이어, 확인되지 않은 과거 AI 응답.\n"
        if speech is not None or narration is not None:
            document += (f"과거 NPC 대사: 「{speech or ''}」\n"
                         f"AI가 붙인 서술(미확정): 「{narration or ''}」\n")
        else:
            document += f"NPC 응답 원문: 「{text}」\n"
        document += ("이렇게 응답했다는 기록이며, 응답 속 행동·주장이나 AI 서술이 "
                     "실제로 일어났다는 엔진 확인은 아닙니다.")
        trust = "unverified_statement"
        speaker, listener, role = npc, "player", "assistant"
    else:
        document = (f"기록 주체=게임 엔진, 청자=NPC {npc}.\n"
                    f"엔진이 확인한 공개 사건: 「{text}」")
        trust = "engine_confirmed"
        speaker = "engine"
        listener, role = npc, "engine"
    # 검색 모듈에서 준비한 CPU E5 모델로 기억 문장을 임베딩합니다.
    vector = embedder.encode(["passage: " + document],
                             normalize_embeddings=True).tolist()
    with _memory_lock:
        # 임베딩 중 초기화된 세션에 뒤늦게 기록이 들어가는 것을 막습니다.
        _memory_scope(session_id, npc)
        _, counters = _npc_memory_scope(session_id, npc)
        current_turn = counters["player_turn"]
        if kind == "npc_statement" and turn_id is not None and turn_id > current_turn:
            raise ValueError("아직 저장되지 않은 사용자 턴에 NPC 응답을 연결할 수 없습니다.")
        turn = (current_turn + 1 if kind == "player_statement"
                else turn_id if kind == "npc_statement" and turn_id is not None
                else current_turn)
        event_index = counters["event_index"] + (1 if kind == "confirmed_event" else 0)
        memory_id = uuid4().hex
        metadata = {"memory_id": memory_id, "session_id": session_id,
                    "npc": npc, "kind": kind, "turn": turn,
                    "event_index": event_index, "trust": trust,
                    "speaker": speaker, "listener": listener, "role": role,
                    "title": (f"플레이어 발언 {turn} / 청자 {npc}" if kind == "player_statement"
                              else f"NPC {npc} 응답 {turn} / 청자 플레이어" if kind == "npc_statement"
                              else f"엔진 공개 사건 {event_index} / 청자 {npc}")}
        if kind == "npc_statement":
            # 원문 채널은 저장소에 보존합니다. 검색 시에는 짧은 본문과 종류를 전달합니다.
            metadata["raw_text"] = text
            if speech is not None:
                metadata["speech"] = speech
            if narration is not None:
                metadata["narration"] = narration
        _memory_store().add(ids=[memory_id], documents=[document],
                            metadatas=[metadata], embeddings=vector)
        if kind == "player_statement":
            counters["player_turn"] = turn
        counters["event_index"] = event_index
    return turn


def remember_player_message(session_id, npc, text, embedder):
    """사용자 발언 원문을 저장하고 해당 NPC의 사용자 턴 번호를 반환합니다.

    기록 종류는 player_statement, 신뢰 구분은 unverified_statement입니다.
    발언자와 청자를 함께 표시해 이후 검색에서 대화 주체를 구분합니다.
    """
    return _remember_memory(session_id, npc, text, embedder, "player_statement")


def remember_npc_message(session_id, npc, text, embedder, turn_id=None,
                         speech=None, narration=None):
    """NPC의 과거 응답을 발화 기록으로 저장합니다. 확정 사건으로 승격하지 않습니다.

    turn_id에 remember_player_message의 반환값을 넣으면 같은 대화 턴으로 묶입니다.
    생략하면 해당 NPC의 마지막 사용자 턴에 연결하며 사용자 턴 수는 늘리지 않습니다.
    """
    return _remember_memory(session_id, npc, text, embedder, "npc_statement",
                            turn_id=turn_id, speech=speech, narration=narration)


def remember_confirmed_event(session_id, npc, text, embedder):
    """엔진이 확인한 공개 사건을 저장하고 현재 사용자 턴 번호를 반환합니다.

    기록 종류는 confirmed_event, 신뢰 구분은 engine_confirmed입니다.
    엔진 사건은 별도의 사건 번호로 관리하며 사용자 발언 번호와 구분합니다.
    """
    return _remember_memory(session_id, npc, text, embedder, "confirmed_event")


def memory_count(session_id, npc):
    """해당 세션·NPC의 발언과 확인된 사건 수를 반환합니다."""
    with _memory_lock:
        npc = _memory_scope(session_id, npc)
        if npc not in _memory_sessions[session_id]["scopes"]:
            return 0
        return len(_memory_store().get(
            where=_memory_where(session_id, npc), include=[])["ids"])


def _memory_preview(document, embedder, max_context_tokens):
    """저장 원문은 유지하고 모델에 전달할 검색 본문만 짧게 만듭니다."""
    tokenizer = getattr(embedder, "tokenizer", None)
    if tokenizer is not None and callable(getattr(tokenizer, "encode", None)):
        token_ids = tokenizer.encode(document, add_special_tokens=False)
        if len(token_ids) > max_context_tokens:
            return (tokenizer.decode(token_ids[:max_context_tokens], skip_special_tokens=True),
                    True, "tokens", max_context_tokens)
        return document, False, "tokens", max_context_tokens
    # 테스트용·기타 임베더에는 토크나이저가 없을 수 있습니다. 토큰 수로 속이지 않습니다.
    character_limit = max_context_tokens * 4
    return document[:character_limit], len(document) > character_limit, "characters", character_limit


def search_memory(session_id, npc, question, embedder, k=3, exclude_recent_turns=6,
                  exclude_recent_turn_ids=None, max_context_tokens=512):
    """같은 세션·NPC의 기억을 검색해 payload용 memory_results 목록을 반환합니다.

    exclude_recent_turn_ids에는 최근 원본 이력의 memory_turn을 넣습니다.
    상태 화면으로 처리된 발언도 중복 검색을 막기 위한 제외 대상입니다.
    빈 목록은 제외할 이력이 없다는 뜻입니다. None일 때만 기존 최근 N턴 방식을 씁니다.
    같은 턴의 사용자·NPC 발화만 제외하고 엔진 사건은 유지합니다.
    저장 원문은 자르지 않으며, 검색 본문만 기본 512토큰으로 제한합니다.
    score는 코사인 유사도이며 정답 확률이 아닙니다.
    """
    if type(k) is not int or k < 0:
        raise ValueError("검색 개수 k는 0 이상의 정수여야 합니다.")
    if type(exclude_recent_turns) is not int or exclude_recent_turns < 0:
        raise ValueError("최근 발언 제외 수는 0 이상의 정수여야 합니다.")
    if type(max_context_tokens) is not int or max_context_tokens < 1:
        raise ValueError("기억 본문 제한은 1 이상의 토큰 수여야 합니다.")
    if exclude_recent_turn_ids is not None:
        if (not isinstance(exclude_recent_turn_ids, (list, tuple, set))
                or any(type(turn) is not int or turn < 0 for turn in exclude_recent_turn_ids)):
            raise ValueError("최근 발언 번호는 0 이상의 정수 목록이어야 합니다.")
        exclude_recent_turn_ids = sorted(set(exclude_recent_turn_ids))
    question = _memory_text(question)
    with _memory_lock:
        npc = _memory_scope(session_id, npc)
        scope = _memory_sessions[session_id]["scopes"].get(npc)
        if scope is None or k == 0:
            return []
        where = _memory_where(session_id, npc)
        turn_filter = None
        if exclude_recent_turn_ids is None:
            cutoff = scope["player_turn"] - exclude_recent_turns
            turn_filter = {"turn": {"$lte": cutoff}}
        elif exclude_recent_turn_ids:
            turn_filter = {"turn": {"$nin": exclude_recent_turn_ids}}
        if turn_filter is not None:
            where = {"$and": [where, {"$or": [
                {"kind": {"$eq": "confirmed_event"}},
                {"$and": [{"kind": {"$in": ["player_statement", "npc_statement"]}},
                          turn_filter]},
            ]}]}
        collection = _memory_store()
        candidates = collection.get(where=where, include=[])["ids"]
        if not candidates:
            return []
    vector = embedder.encode(["query: " + question],
                             normalize_embeddings=True).tolist()
    with _memory_lock:
        _memory_scope(session_id, npc)
        result = collection.query(
            query_embeddings=vector, n_results=min(k, len(candidates)),
            where=where, include=["documents", "metadatas", "distances"])
    memories = []
    for text, metadata, distance in zip(result["documents"][0],
                                         result["metadatas"][0], result["distances"][0]):
        # 모델용 메타데이터는 세션 필터 ID를 제외한 항목으로 구성합니다.
        metadata = {key: value for key, value in metadata.items()
                    if key not in {"session_id", "raw_text", "speech", "narration"}}
        text, truncated, limit_type, limit = _memory_preview(text, embedder, max_context_tokens)
        metadata.update({"text_truncated": truncated, "context_limit_type": limit_type,
                         "context_limit": limit})
        memories.append({"source": "memory", "text": text,
                         "score": round(1.0 - float(distance), 6),
                         "metadata": metadata})
    return memories


def discard_memory_session(session_id):
    """세션에 속한 모든 NPC 기억을 지우고 ID를 폐기합니다."""
    with _memory_lock:
        _memory_scope(session_id, "")
        for npc in _memory_sessions[session_id]["scopes"]:
            if _memory_collection is not None:
                _memory_collection.delete(where=_memory_where(session_id, npc))
        del _memory_sessions[session_id]



## 3. 대화 두 턴과 엔진 사건 저장

`remember_player_message()`가 반환한 번호를 같은 턴의 NPC 응답에도 붙입니다.
NPC 대사와 AI 서술은 각각 보존하지만 실제 행동 완료로 취급하지 않습니다.
아래 엔진 사건도 실습을 위해 작성한 가상 입력입니다.


In [ ]:
session_a = create_memory_session()
npc_label = "기록원"
history = []
dialogue = [
    ("청동 열쇠는 낡은 지도 주머니에 있어요.", "돌아오시면 그 주머니부터 함께 살펴보죠."),
    ("동쪽 문을 열어볼게요.", "문 앞에서 기다리겠습니다."),
]
for player_text, npc_speech in dialogue:
    turn = remember_player_message(session_a, npc_label, player_text, embedder)
    remember_npc_message(session_a, npc_label, npc_speech, embedder,
                         turn_id=turn, speech=npc_speech, narration="")
    history.extend([
        {"role": "user", "content": player_text, "memory_turn": turn},
        {"role": "assistant", "content": npc_speech, "memory_turn": turn,
         "speech": npc_speech, "narration": ""},
    ])
save_dialogue_history(session_a, npc_label, history)
remember_confirmed_event(session_a, npc_label, "동쪽 문이 열렸습니다.", embedder)
assert memory_count(session_a, npc_label) == 5
print("사용자 발언 2개 + NPC 응답 2개 + 엔진 사건 1개 저장")


## 4. 전체 기억과 자료 종류 확인

`exclude_recent_turn_ids=[]`는 전달할 최근 이력이 없다는 뜻이므로 어떤 발화도 제외하지 않습니다.
본문과 함께 화자·종류·신뢰도·턴을 확인합니다. 저장 원문은 유지하고 검색 본문만 기본 512토큰으로 제한합니다.
잘린 경우 `text_truncated`가 표시됩니다. 자료 번호나 유사도만으로 내용이 참이라고 판단하지 않습니다.


In [ ]:
def show_memory_rows(rows):
    for index, row in enumerate(rows, 1):
        meta = row["metadata"]
        print(f"[M{index}] {meta['kind']} / {meta['speaker']} / "
              f"{meta['trust']} / turn={meta['turn']} / score={row['score']:.4f}")
        print(row["text"])
        print("본문 잘림:", meta["text_truncated"])
        print()

question = "열쇠를 어디서 함께 살펴보기로 했나요?"
all_rows = search_memory(session_a, npc_label, question, embedder,
                         k=10, exclude_recent_turn_ids=[])
assert len(all_rows) == 5
assert any(row["metadata"]["kind"] == "npc_statement" for row in all_rows)
assert all(row["metadata"]["trust"] == "unverified_statement"
           for row in all_rows if row["metadata"]["kind"] == "npc_statement")
show_memory_rows(all_rows)


## 5. 실제 전달한 최근 턴만 제외

출력 프로토타입은 최근 최대 12메시지를 직접 전달하고 그 밖의 기억을 보충합니다.
아래에서는 최근 이력에 2턴만 전달한다고 가정합니다. `memory_turn=2`인 사용자·NPC 발화만 제외합니다.
엔진의 확정 사건은 최근 턴에 속해도 검색에서 제외하지 않습니다.
기존 `exclude_recent_turns` 옵션도 남아 있지만 실제 출력 연결은 턴 번호 목록을 사용합니다.


In [ ]:
recent_history = history[-2:]
recent_turn_ids = sorted({item["memory_turn"] for item in recent_history})
older_rows = search_memory(session_a, npc_label, question, embedder,
                           k=10, exclude_recent_turn_ids=recent_turn_ids)
assert len(older_rows) == 3
assert all(row["metadata"]["turn"] != 2 or
           row["metadata"]["kind"] == "confirmed_event" for row in older_rows)
show_memory_rows(older_rows)


## 6. NPC 왕복과 세션 분리

NPC를 바꿀 때 세션을 폐기하지 않고 해당 인물의 이력을 불러옵니다.
다른 인물에게 다녀와도 원래 인물의 대화가 남습니다. 다른 접속자의 세션은 별도로 검색합니다.
아래 검사는 데이터 분리를 확인하는 예제이며 의미 검색 정확도 평가는 아닙니다.


In [ ]:
other_npc = "문지기"
save_dialogue_history(session_a, other_npc, [{"role": "user", "content": "안녕하세요."}])
restored = load_dialogue_history(session_a, npc_label)
assert restored == history
restored.clear()
assert len(load_dialogue_history(session_a, npc_label)) == 4
assert search_memory(session_a, other_npc, question, embedder,
                     exclude_recent_turn_ids=[]) == []

session_b = create_memory_session()
assert load_dialogue_history(session_b, npc_label) == []
assert search_memory(session_b, npc_label, question, embedder,
                     exclude_recent_turn_ids=[]) == []
remember_player_message(session_b, npc_label, "다른 플레이어의 가상 발언입니다.", embedder)
assert memory_count(session_b, npc_label) == 1
assert memory_count(session_a, npc_label) == 5
print("NPC 왕복 이력·접속자별 기억 분리 확인")


## 7. 초기화와 임시 저장의 범위

전체 초기화는 세션의 모든 NPC 기억과 이력을 폐기합니다. 폐기한 ID는 다시 쓰지 않습니다.
다른 세션의 기억은 지우지 않으며 런타임이 종료되면 임시 기억 전체가 사라집니다.
이 실습은 저장·검색·분리 동작을 설명하며 답변의 자연스러움이나 사실성을 보장하지 않습니다.


In [ ]:
old_session = session_a
discard_memory_session(old_session)
try:
    load_dialogue_history(old_session, npc_label)
except ValueError:
    print("폐기한 세션 사용 차단 확인")
else:
    raise AssertionError("폐기한 세션은 다시 쓸 수 없어야 합니다.")
assert memory_count(session_b, npc_label) == 1
session_a = create_memory_session()
assert memory_count(session_a, npc_label) == 0
discard_memory_session(session_b)
print("해당 세션만 초기화하고 새 세션 준비 완료")


## 실행 링크

- [프로토타입 실행 — Colab](https://colab.research.google.com/github/bjh281/MY-TRPG/blob/main/BAE/Output_Code.ipynb)
- [기억 RAG 실험 — Colab](https://colab.research.google.com/github/bjh281/MY-TRPG/blob/main/BAE/notebooks/Memory_RAG.ipynb)
